# `geoai-datacubes` -- Building detection with YOLO on NAIP

<a href="https://colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/main/notebooks/03_building_detection.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

**Welcome.** This is the third pedagogical notebook in the `geoai-datacubes`
trilogy. The first (`00_geoai_datacubes_tour.ipynb`) walks you through the
data-acquisition side of the pipeline; the second
(`01_water_classification.ipynb`) trains four standard classifiers for a
binary **semantic-segmentation** target (every pixel gets a class label).

This notebook switches the modelling problem from *labelling pixels* to
**detecting objects** -- drawing one bounding box around each individual
building. Object detection is the right framing whenever the downstream
question is *"how many of X are there, and where exactly?"* rather than
*"what fraction of the scene is X?"*. Damage-assessment after a hurricane,
parking-lot counting, illegal-construction monitoring, ship spotting in
ports: all of these are object detection, not segmentation.

Concretely you will:

1. Fetch NAIP (1 m public-domain US aerial imagery) over Columbus, Cincinnati,
   and Cleveland through the pipeline's new `NAIP` mission profile.
2. Download the Microsoft US Building Footprints dataset for Ohio (a free,
   permissively-licensed polygon ground truth) and clip it to each AOI.
3. Convert the polygon ground truth into **YOLO-format bounding-box
   labels** -- the boring-but-critical glue step that makes any
   detector trainable.
4. Train a tiny **YOLOv8n** model (the "nano" Ultralytics variant -- 3.2 M
   parameters) for ~60 epochs on CPU and report mAP@0.5, mAP@0.5:0.95,
   precision, and recall.
5. Sanity-check the result visually with a side-by-side
   ground-truth-vs-prediction panel.
6. Stop in the middle for a **resolution-comparison sidebar**: the same
   neighbourhood at 1 m (NAIP) vs 10 m (Sentinel-2), because the single
   most important reason object detection on satellite imagery fails is
   that the operator pointed it at imagery whose pixels are larger than
   the objects of interest.

Runs end-to-end on a Colab CPU in roughly 15-30 minutes; most of the time
is spent waiting for NAIP to download and YOLO to converge.

> *Why buildings?* Buildings are a great pedagogical target because the
> visual signal (rectangular blob with a clear shadow on one side) is
> unambiguous to a human reader, ground-truth polygons are freely
> available state-wide, and the object scale (~10 x 10 m) sits right at the
> boundary where imagery resolution starts to matter. That last property
> is what lets the resolution-comparison sidebar land hard.

## 1. Setup

Standard Colab / local bootstrap. On Colab, this clones the repo into
`/content/geoai-datacubes` and pip-installs the handful of extras Colab
does not ship with (most importantly `ultralytics`, the Python package
that fronts the YOLO models we will train). Locally it is a no-op that
just confirms the layout.

In [ ]:
# --- Colab / local bootstrap ---
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected -- bootstrapping repo + dependencies")
    REPO_DIR = Path("/content/geoai-datacubes")
    if not REPO_DIR.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/buckai-observatory/geoai-datacubes.git",
            str(REPO_DIR),
        ])
    else:
        print(f"repo already cloned at {REPO_DIR}")

    # Colab pre-installs rasterio / shapely / scikit-image / scipy / tqdm /
    # pandas / pyproj / requests / matplotlib / scikit-learn / torch.
    # These extras may be missing for this notebook:
    missing = []
    for pkg, importname in [("pystac",      "pystac"),
                            ("contextily",  "contextily"),
                            ("geopandas",   "geopandas"),
                            ("ultralytics", "ultralytics")]:
        try:
            __import__(importname)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"pip install -q {' '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    os.chdir(REPO_DIR / "notebooks")
    print(f"cwd = {os.getcwd()}")
else:
    print("Local environment -- using existing checkout")

Imports, `sys.path` setup so `modules/sentinel_pipeline` is reachable,
and a scratch folder `notebooks/_outputs_obj/` for everything this
notebook writes (gitignored).

In [ ]:
# --- imports + path setup ---
import glob, os, sys, json, shutil, time, math, warnings, random, zipfile, io, re
from pathlib import Path

# Keep CPU threading conservative -- YOLO + sklearn + matplotlib coexist
# better with a small thread pool on a laptop CPU.
os.environ.setdefault("OMP_NUM_THREADS",      "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS",      "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS",  "2")

import numpy as np
import pandas as pd
import rasterio
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

NB_DIR = Path.cwd()
if NB_DIR.name != "notebooks":
    for p in (NB_DIR, *NB_DIR.parents):
        if (p / "notebooks").is_dir() and (p / "geoai_datacubes").is_dir():
            NB_DIR = p / "notebooks"
            break
REPO_ROOT = NB_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

OUT       = NB_DIR / "_outputs_obj"
NAIP_DIR  = OUT / "naip"
S2_DIR    = OUT / "s2"
TILES_DIR = OUT / "tiles"
LBL_DIR   = OUT / "labels"
YOLO_DIR  = OUT / "yolo"
RUNS_DIR  = OUT / "runs"
for d in (OUT, NAIP_DIR, S2_DIR, TILES_DIR, LBL_DIR, YOLO_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"]  = 90
plt.rcParams["savefig.dpi"] = 90
plt.rcParams["axes.grid"]   = False
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Repo root :", REPO_ROOT)
print("Package   :", REPO_ROOT / "geoai_datacubes")
print("Outputs   :", OUT)

Pull in the four pieces of the pipeline this notebook actually uses --
the AOI resolver, the mission table (so we can read NAIP and Sentinel-2's
profiles), the fetcher itself, and PyTorch / Ultralytics.

In [ ]:
from geoai_datacubes.fetch import resolve_aoi
from geoai_datacubes.fetch import MISSION_PROFILES, get_profile
from geoai_datacubes.fetch import fetch_sentinel_data

import torch
from ultralytics import YOLO
torch.manual_seed(SEED)
torch.set_num_threads(2)

print("Available missions:", sorted(MISSION_PROFILES.keys()))
print("torch:", torch.__version__, "(device: cpu)")
from ultralytics import __version__ as ULTRA_VER
print("ultralytics:", ULTRA_VER)

import geopandas as gpd
import shapely
from shapely.geometry import box as sh_box, Polygon, MultiPolygon
print("geopandas:", gpd.__version__, "   shapely:", shapely.__version__)

## 2. USER INPUT

Three AOIs, one per city, with the same lat/lon centres notebook 01 uses.
NAIP at 1 m delivers about a million pixels per square mile, so we keep
each AOI a bit smaller than the 4 mi squares of notebook 01 -- a 5 mi
side is generous enough to give YOLO a few hundred 512 m tiles per city
(comfortably above the small-dataset floor where YOLO bounces around at
random) while still decompressing the COGs in a few minutes per city on
a typical connection.

The time window covers 2022-2024 so we pick up the latest Ohio NAIP
acquisition (2023 in this case).

In [ ]:
# ============================================================
# USER INPUT
# ============================================================
CITY_AOIS = {
    "columbus":   {"center": (40.0067, -83.0305), "side_miles": 5.0, "role": "train"},
    "cincinnati": {"center": (39.0997, -84.5147), "side_miles": 5.0, "role": "val"},
    "cleveland":  {"center": (41.4993, -81.6944), "side_miles": 5.0, "role": "test"},
}

for city, spec in CITY_AOIS.items():
    spec["bbox"] = resolve_aoi({"center": spec["center"],
                                "side_miles": spec["side_miles"]})

# Time window -- latest available NAIP acquisition for Ohio is 2023.
TIME_RANGE_NAIP = ("2022-01-01", "2024-12-31")

# Tile geometry -- 512 x 512 px tiles at 1 m GSD = 512 m square ground footprint.
TILE_PX  = 512
NAIP_RES = 1.0   # metres per pixel

for city, spec in CITY_AOIS.items():
    bb = spec["bbox"]
    h_km = (bb[3] - bb[1]) * 111.0
    w_km = (bb[2] - bb[0]) * 111.0 * math.cos(math.radians(bb[1]))
    print(f"{city:11s}  role={spec['role']:5s}  side={spec['side_miles']} mi"
          f"  bbox={[round(x,3) for x in bb]}  ({h_km:.2f} x {w_km:.2f} km)")
print(f"\nTime window     : {TIME_RANGE_NAIP[0]} .. {TIME_RANGE_NAIP[1]}")
print(f"Tile geometry   : {TILE_PX} x {TILE_PX} px at {NAIP_RES} m -> "
      f"{TILE_PX*NAIP_RES:.0f} m square per tile")

## 3. Fetch NAIP for each city

NAIP comes through Microsoft Planetary Computer, no credentials needed.
The dispatcher routes `mission="NAIP"` to `provider="planetary_computer"`
automatically. The fetcher writes one `<Mission>_full_size.tiff` per scene
under `_outputs_obj/naip/`.

NAIP is delivered as a **single multi-band COG** with R / G / B / NIR in
bands 1-4. The pipeline's `asset_map` uses `(asset_key, band_index)`
tuples to reach into that one asset; everything else in the fetch flow
works just like Sentinel-2.

In [ ]:
def fetch_naip_for_city(city, bbox):
    print(f"\n=== NAIP fetch for {city} ===")
    t0 = time.time()
    data, bands = fetch_sentinel_data(
        mission="NAIP",
        bands=["R", "G", "B", "NIR"],
        time_range=TIME_RANGE_NAIP,
        roi=bbox,
        resolution=NAIP_RES,
        save_folder=str(NAIP_DIR),
        max_cloud_coverage=1.0,   # NAIP profile sets cloud_filter=False; this is a no-op
    )
    dt = time.time() - t0
    return dt

naip_times = {}
for city, spec in CITY_AOIS.items():
    naip_times[city] = fetch_naip_for_city(city, spec["bbox"])

print()
for city, dt in naip_times.items():
    print(f"  {city:11s}  fetched in {dt:5.1f} s")

Locate the per-city scene folder, read a few key fields from the
GeoTIFF (CRS, pixel grid size, acquisition date encoded in the scene id),
and print a summary line. We will reuse `city_scene[city]` throughout.

In [ ]:
def _latest_naip_scene_dir(city_bbox):
    # The pipeline writes one folder per scene under NAIP_DIR; pick the
    # folder whose footprint covers (or best overlaps) the AOI centre.
    candidates = sorted(NAIP_DIR.glob("NAIP_*"))
    best = None; best_overlap = -1.0
    cx = 0.5 * (city_bbox[0] + city_bbox[2])
    cy = 0.5 * (city_bbox[1] + city_bbox[3])
    for d in candidates:
        tif = d / "NAIP_full_size.tiff"
        if not tif.exists():
            continue
        with rasterio.open(tif) as src:
            from rasterio.warp import transform_bounds
            lb = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
            if lb[0] <= cx <= lb[2] and lb[1] <= cy <= lb[3]:
                # AOI centre inside this scene; prefer it
                return d
            ix0 = max(lb[0], city_bbox[0]); iy0 = max(lb[1], city_bbox[1])
            ix1 = min(lb[2], city_bbox[2]); iy1 = min(lb[3], city_bbox[3])
            overlap = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
            if overlap > best_overlap:
                best_overlap = overlap; best = d
    return best

city_scene = {}
for city, spec in CITY_AOIS.items():
    sd = _latest_naip_scene_dir(spec["bbox"])
    if sd is None:
        raise RuntimeError(f"No NAIP scene folder found for {city}")
    city_scene[city] = sd

print(f"{'city':11s} {'date':10s} {'crs':10s} {'pixels':>15s}  scene_id")
print("-" * 90)
for city, sd in city_scene.items():
    tif = sd / "NAIP_full_size.tiff"
    with rasterio.open(tif) as src:
        h, w = src.shape
        crs  = str(src.crs)
    scene_id = sd.name.split("_", 2)[-1]
    date_str = sd.name.split("_")[1]
    pixels = f"{h} x {w}"
    print(f"{city:11s} {date_str:10s} {crs:10s} {pixels:>15s}  {scene_id}")

## 4. Download Microsoft US Building Footprints (Ohio)

The ground truth comes from Microsoft's open
[US Building Footprints](https://github.com/microsoft/USBuildingFootprints) --
machine-extracted building outlines covering the entire US, released under
the **Open Data Commons Open Database Licence (ODbL)** so we are allowed
to redistribute derived products. We grab the Ohio-only file from the
canonical mirror.

The full state file is ~180 MB zipped / ~1.4 GB unzipped (~5.5 million
polygons), so reading it naively into geopandas blows up memory. Instead
we **stream-decompress it line-by-line**, do a cheap regex bbox prefilter
to discard polygons obviously outside our three AOIs, and only hand the
small surviving subset to geopandas. This finishes in ~15 s on a laptop.

In [ ]:
OHIO_URL  = "https://minedbuildings.z5.web.core.windows.net/legacy/usbuildings-v2/Ohio.geojson.zip"
OHIO_ZIP  = OUT / "Ohio.geojson.zip"

if not OHIO_ZIP.exists():
    print(f"Downloading {OHIO_URL}")
    t0 = time.time()
    with requests.get(OHIO_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length") or 0)
        written = 0
        with open(OHIO_ZIP, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
                written += len(chunk)
        print(f"  downloaded {written/1024/1024:.1f} MB in {time.time()-t0:.1f} s")
else:
    print(f"  using cached {OHIO_ZIP} ({OHIO_ZIP.stat().st_size/1024/1024:.1f} MB)")

Build a single union bbox covering all three city AOIs (with a small
pad) so the streaming prefilter only has to do one cheap test per feature.
Then read the file feature-by-feature, dropping any polygon whose first
vertex is outside that union -- this discards roughly 99.9% of the
state-wide records without any JSON parsing.

In [ ]:
def _union_bbox(aois, pad_deg=0.02):
    bbs = [s["bbox"] for s in aois.values()]
    return (min(b[0] for b in bbs) - pad_deg,
            min(b[1] for b in bbs) - pad_deg,
            max(b[2] for b in bbs) + pad_deg,
            max(b[3] for b in bbs) + pad_deg)

UB = _union_bbox(CITY_AOIS)
print(f"Union bbox (with pad): {[round(x,3) for x in UB]}")

# Stream-extract candidate features
print(f"Streaming-filtering {OHIO_ZIP.name} into the AOI union bbox ...")
t0 = time.time()
features = []
with zipfile.ZipFile(OHIO_ZIP) as z:
    with z.open("Ohio.geojson") as raw:
        reader = io.TextIOWrapper(raw, encoding="utf-8")
        # MS USBF v2 ships as a FeatureCollection with one Feature per line.
        coord0 = re.compile(r'\[\[([-0-9.]+),([-0-9.]+)\]')
        for line in reader:
            s = line.strip().rstrip(",")
            if not s.startswith('{"type":"Feature"'):
                continue
            m = coord0.search(s)
            if not m:
                continue
            lon = float(m.group(1)); lat = float(m.group(2))
            if not (UB[0] <= lon <= UB[2] and UB[1] <= lat <= UB[3]):
                continue
            try:
                features.append(json.loads(s))
            except Exception:
                continue
elapsed = time.time() - t0
print(f"  kept {len(features):,} candidate polygons in {elapsed:.1f} s "
      f"(out of ~5.5M state-wide)")

fc = {"type": "FeatureCollection", "features": features}
gdf_all = gpd.GeoDataFrame.from_features(fc, crs="EPSG:4326")
print(f"  geopandas frame: {len(gdf_all):,} rows, crs={gdf_all.crs}")

Reproject the candidate frame to NAIP's CRS (EPSG:26917, UTM zone 17N
covers all of Ohio). Then clip per city AOI so each city has its own
clean footprint set, and report counts so the reader can sanity-check
that the regex prefilter didn't drop anything.

In [ ]:
gdf_utm = gdf_all.to_crs("EPSG:26917")
print(f"reprojected -> {gdf_utm.crs}")

city_footprints = {}
city_aoi_utm    = {}   # AOI bbox in UTM 17N (as a shapely polygon)
for city, spec in CITY_AOIS.items():
    bb_ll = spec["bbox"]
    aoi_ll = gpd.GeoSeries([sh_box(*bb_ll)], crs="EPSG:4326").to_crs("EPSG:26917")
    aoi_utm = aoi_ll.iloc[0]
    city_aoi_utm[city] = aoi_utm
    sub = gdf_utm[gdf_utm.intersects(aoi_utm)].copy()
    sub["geometry"] = sub.geometry.intersection(aoi_utm)
    sub = sub[~sub.geometry.is_empty]
    city_footprints[city] = sub
    print(f"  {city:11s}: {len(sub):>6,} buildings in AOI")

## 5. Visualise the ground truth

For each city we show the NAIP RGB composite for the AOI extent with the
clipped building footprints overlaid in red. This is the "look how dense
the labels are" sanity check -- if the overlay shows buildings landing in
the streets, we have a CRS or coordinate-order bug to fix before going
any further.

In [ ]:
def _read_aoi_rgb(scene_dir, bbox_ll, max_side=1200):
    """Read the AOI window from a NAIP scene and return (rgb, aoi_utm_bbox)."""
    tif = scene_dir / "NAIP_full_size.tiff"
    with rasterio.open(tif) as src:
        from rasterio.warp import transform_bounds
        # Reproject AOI corners to the scene's CRS
        utm_bb = transform_bounds("EPSG:4326", src.crs, *bbox_ll)
        win = rasterio.windows.from_bounds(*utm_bb, transform=src.transform)
        win = win.round_offsets().round_lengths()
        rgb = src.read([1, 2, 3], window=win)  # R, G, B  (3,H,W)
        # NAIP is uint8 stored as float32 in the cube; clip and scale
        rgb = np.clip(rgb, 0, 255).astype(np.uint8)
        rgb = np.transpose(rgb, (1, 2, 0))     # (H,W,3)
        if max(rgb.shape[:2]) > max_side:
            step = max(1, max(rgb.shape[:2]) // max_side)
            rgb = rgb[::step, ::step]
        # Return UTM bbox so the polygon overlay lines up
        return rgb, utm_bb

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, (city, spec) in zip(axes, CITY_AOIS.items()):
    rgb, utm_bb = _read_aoi_rgb(city_scene[city], spec["bbox"])
    ax.imshow(rgb, extent=(utm_bb[0], utm_bb[2], utm_bb[1], utm_bb[3]),
              origin="upper")
    sub = city_footprints[city]
    if len(sub) > 0:
        sub.boundary.plot(ax=ax, color="red", linewidth=0.6)
    ax.set_title(f"{city.title()} ({spec['role']})\n"
                 f"{len(sub):,} building footprints, {spec['side_miles']} mi AOI")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 6. Convert footprints to YOLO labels (the boring but critical step)

YOLO expects a very specific on-disk layout:

```
yolo/
  images/train/   <tile>.png
  images/val/
  images/test/
  labels/train/   <tile>.txt
  labels/val/
  labels/test/
```

with `<tile>.txt` carrying one line per object,

```
<class_id> <cx> <cy> <w> <h>
```

with all four coordinates **normalised to `[0, 1]` relative to the image
size**. So for every fixed-size tile we cut from a NAIP scene, we need
to:

1. Compute the tile's geographic bbox in NAIP's UTM.
2. Intersect the city footprints with that bbox.
3. For each intersected polygon, compute its axis-aligned bbox and convert
   to YOLO's normalised `(cx, cy, w, h)` form.
4. Drop any polygon whose final bbox is too small to detect (<6 px at 1 m
   means a < 6 m wide building, which is below the YOLO floor anyway).

The tile cutter and the label writer live in one helper so it is obvious
that the two stay in sync.

Cities are mapped to YOLO splits using the same role plan as notebook 01:
**Columbus -> train, Cincinnati -> val, Cleveland -> test**. This is the
hardest of the four split strategies the pipeline supports (cross-city
generalisation) and pulls no punches on the model.

In [ ]:
from PIL import Image

CLASS_ID = 0          # single class: "building"
MIN_BOX_PX = 6        # drop boxes smaller than this many px on either side

def _polygon_axis_bbox(geom):
    """Return (xmin, ymin, xmax, ymax) for any polygonal geometry, else None."""
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "GeometryCollection":
        # keep only polygonal sub-parts
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        if not polys:
            return None
        # union
        from shapely.ops import unary_union
        geom = unary_union(polys)
        if geom.is_empty:
            return None
    return geom.bounds  # (minx, miny, maxx, maxy)


def tile_and_label_city(city, split, stride=None):
    """Cut TILE_PX x TILE_PX tiles from the AOI of `city` and write the
    matching YOLO label files. `stride` defaults to TILE_PX (no overlap);
    pass a smaller value to densify the tile grid (useful for the train
    split, where more views per scene gives YOLO more chances to see each
    building under different translations). Returns the list of tile
    basenames."""
    scene_dir = city_scene[city]
    spec = CITY_AOIS[city]
    tif = scene_dir / "NAIP_full_size.tiff"

    img_out = YOLO_DIR / "images" / split
    lbl_out = YOLO_DIR / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    fps = city_footprints[city]
    fp_sindex = fps.sindex if len(fps) > 0 else None

    with rasterio.open(tif) as src:
        from rasterio.warp import transform_bounds
        # AOI bbox in the scene's CRS
        utm_bb = transform_bounds("EPSG:4326", src.crs, *spec["bbox"])
        # Pixel window for the AOI in the scene grid
        win = rasterio.windows.from_bounds(*utm_bb, transform=src.transform)
        win = win.round_offsets().round_lengths()
        # Read the AOI sub-array once (R/G/B/NIR) - much faster than per-tile reads
        arr = src.read([1, 2, 3, 4], window=win)   # (4, H, W) float32 0..255
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        # Compute the AOI window's transform (top-left UTM corner + pixel size)
        win_transform = src.window_transform(win)

    H, W = arr.shape[1], arr.shape[2]
    px = NAIP_RES   # m per pixel
    print(f"  {city:11s} ({split:5s})  AOI raster: {H} x {W} px"
          f" -> {(H*W*4)/1024/1024:.1f} MB")

    # Tile grid (skip the trailing partial tile to keep all tiles same size)
    if stride is None:
        stride = TILE_PX
    rows = list(range(0, H - TILE_PX + 1, stride))
    cols = list(range(0, W - TILE_PX + 1, stride))
    tile_names = []

    n_tiles_with_boxes = 0
    n_total_boxes = 0

    for r in rows:
        for c in cols:
            sub = arr[:3, r:r+TILE_PX, c:c+TILE_PX]   # RGB only for YOLO inputs
            # Drop tiles whose AOI clip is mostly out-of-bounds (rare here,
            # since we computed `win` from the AOI bbox).
            tile_img = np.transpose(sub, (1, 2, 0))   # (H, W, 3)

            # Tile bbox in scene CRS
            tile_minx = win_transform.c + c * win_transform.a
            tile_maxy = win_transform.f + r * win_transform.e   # e is negative
            tile_maxx = tile_minx + TILE_PX * win_transform.a
            tile_miny = tile_maxy + TILE_PX * win_transform.e
            tile_poly = sh_box(min(tile_minx, tile_maxx), min(tile_miny, tile_maxy),
                               max(tile_minx, tile_maxx), max(tile_miny, tile_maxy))

            lines = []
            if fp_sindex is not None:
                idx_hits = list(fp_sindex.intersection(tile_poly.bounds))
                for idx in idx_hits:
                    geom = fps.geometry.iloc[idx]
                    inter = geom.intersection(tile_poly)
                    bb = _polygon_axis_bbox(inter)
                    if bb is None:
                        continue
                    minx, miny, maxx, maxy = bb
                    # Convert to pixel coordinates *within this tile*.
                    # Image origin (row 0) is at the TOP, so y axis flips.
                    x0_px = (minx - tile_poly.bounds[0]) / px
                    x1_px = (maxx - tile_poly.bounds[0]) / px
                    y0_px = (tile_poly.bounds[3] - maxy) / px
                    y1_px = (tile_poly.bounds[3] - miny) / px
                    w_px = x1_px - x0_px
                    h_px = y1_px - y0_px
                    if w_px < MIN_BOX_PX or h_px < MIN_BOX_PX:
                        continue
                    # Clip to tile, then normalise.
                    x0_px = max(0.0, x0_px); y0_px = max(0.0, y0_px)
                    x1_px = min(float(TILE_PX), x1_px); y1_px = min(float(TILE_PX), y1_px)
                    cx = 0.5 * (x0_px + x1_px) / TILE_PX
                    cy = 0.5 * (y0_px + y1_px) / TILE_PX
                    w  = (x1_px - x0_px) / TILE_PX
                    h  = (y1_px - y0_px) / TILE_PX
                    if w <= 0 or h <= 0:
                        continue
                    lines.append(f"{CLASS_ID} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            base = f"{city}_r{r:05d}_c{c:05d}"
            Image.fromarray(tile_img).save(img_out / f"{base}.png")
            (lbl_out / f"{base}.txt").write_text("\n".join(lines))
            tile_names.append(base)
            if lines:
                n_tiles_with_boxes += 1
                n_total_boxes += len(lines)

    print(f"     {len(tile_names)} tiles, {n_tiles_with_boxes} with >=1 box,"
          f" {n_total_boxes} boxes total")
    return tile_names

# Wipe any previous YOLO layout, then rebuild
for d in ("images", "labels"):
    p = YOLO_DIR / d
    if p.exists():
        shutil.rmtree(p)

# Train: 50%-overlap stride so the tiny training city yields ~4x more
# views per AOI (this is purely sliding-window augmentation; each pixel
# is now seen in up to 4 different tile positions).
# Val and test: no overlap -- we want clean, independent metrics.
tiles_by_city = {
    "columbus":   tile_and_label_city("columbus",   "train", stride=TILE_PX // 2),
    "cincinnati": tile_and_label_city("cincinnati", "val"),
    "cleveland":  tile_and_label_city("cleveland",  "test"),
}

print("\nWrote YOLO layout under:", YOLO_DIR)

## 7. Visualise tiles with their YOLO boxes overlaid

Quick sanity check that the polygon -> bbox -> YOLO-normalised conversion
landed correctly. We pick four random tiles, one each from train / val
(plus two more from train so the panel has variety), draw the YOLO boxes
back over the image, and we should see neat rectangles around buildings,
**not** around lawns or driveways. If the boxes are systematically offset
or rotated, fix the conversion before training -- garbage labels train
garbage models.

In [ ]:
def _draw_yolo_boxes(ax, img, lbl_path, color="lime"):
    ax.imshow(img)
    if not Path(lbl_path).exists():
        return 0
    lines = Path(lbl_path).read_text().splitlines()
    H, W = img.shape[:2]
    n = 0
    for ln in lines:
        parts = ln.split()
        if len(parts) != 5:
            continue
        _cls, cx, cy, w, h = parts
        cx = float(cx) * W; cy = float(cy) * H
        w  = float(w)  * W; h  = float(h)  * H
        x0 = cx - w/2; y0 = cy - h/2
        ax.add_patch(Rectangle((x0, y0), w, h, fill=False, edgecolor=color, linewidth=1.5))
        n += 1
    return n

# Pick tiles that actually have buildings on them, so the panel is informative.
rng = random.Random(SEED)
def _pick_with_boxes(split, k):
    img_dir = YOLO_DIR / "images" / split
    lbl_dir = YOLO_DIR / "labels" / split
    pool = []
    for img in sorted(img_dir.glob("*.png")):
        lbl = lbl_dir / (img.stem + ".txt")
        if lbl.exists() and lbl.stat().st_size > 0:
            pool.append(img)
    rng.shuffle(pool)
    return pool[:k]

samples = (
    [(p, "train") for p in _pick_with_boxes("train", 2)] +
    [(p, "val")   for p in _pick_with_boxes("val",   1)] +
    [(p, "test")  for p in _pick_with_boxes("test",  1)]
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (p, split) in zip(axes, samples):
    img = np.asarray(Image.open(p))
    lbl = YOLO_DIR / "labels" / split / (p.stem + ".txt")
    n = _draw_yolo_boxes(ax, img, lbl, color="lime")
    ax.set_title(f"{split}: {p.stem}\n{n} YOLO boxes")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 8. Resolution sidebar: NAIP (1 m) vs Sentinel-2 (10 m)

Before we train anything, let's make the case that NAIP is the right
imagery source visually. We fetch a Sentinel-2 L2A scene over the
**same Columbus AOI**, snap one neighbourhood block out of each scene,
and draw the same building's footprint as a red box on both panels.

A typical residential building is roughly 10 x 10 m. At 1 m NAIP that
covers ~10 x 10 pixels. At 10 m Sentinel-2 that covers ~1 x 1 pixel --
literally a single sensor cell. YOLO's small-object floor is around
16 x 16 px, with practical reliability not kicking in until ~32 x 32 px.
NAIP comfortably clears the practical floor for residential buildings;
S2 falls below the hard floor.

(We tolerate clouds liberally here -- a Sentinel-2 scene's eo:cloud_cover
includes clouds *anywhere in the granule*, so a fairly cloudy granule
can still have a clean AOI window.)

In [ ]:
# Fetch S2 L2A over the Columbus AOI for the same time window
print("Fetching Sentinel-2 L2A over the Columbus AOI (visual band only) ...")
t0 = time.time()
s2_data, s2_bands = fetch_sentinel_data(
    mission="Sentinel-2",
    bands=["B04", "B03", "B02"],
    time_range=("2023-05-01", "2023-09-30"),
    roi=CITY_AOIS["columbus"]["bbox"],
    resolution=10,
    save_folder=str(S2_DIR),
    max_cloud_coverage=0.30,
)
print(f"  fetched in {time.time()-t0:.1f} s")

# Find that scene
s2_scenes = sorted(S2_DIR.glob("S2*")) + sorted(S2_DIR.glob("Sentinel*"))
# fetch_sentinel_data writes folders named "<Mission>_<date>_<scene_id>"
s2_scene_dir = sorted([d for d in S2_DIR.iterdir() if d.is_dir()])[-1]
s2_tif = s2_scene_dir / "Sentinel-2_full_size.tiff"
print(f"  scene: {s2_scene_dir.name}")

In [ ]:
# Pick a small neighbourhood (300 x 300 m) near the AOI centre that has
# at least one well-defined building on both panels.
city = "columbus"
bb_ll = CITY_AOIS[city]["bbox"]
naip_tif = city_scene[city] / "NAIP_full_size.tiff"

# Window the same 300 m x 300 m patch out of each
NEIGH_M = 300.0
cx_ll = 0.5 * (bb_ll[0] + bb_ll[2])
cy_ll = 0.5 * (bb_ll[1] + bb_ll[3])
# Convert centre to UTM via geopandas
ctr_utm = gpd.GeoSeries([gpd.points_from_xy([cx_ll], [cy_ll])[0]],
                        crs="EPSG:4326").to_crs("EPSG:26917").iloc[0]
neigh_utm = (ctr_utm.x - NEIGH_M/2, ctr_utm.y - NEIGH_M/2,
             ctr_utm.x + NEIGH_M/2, ctr_utm.y + NEIGH_M/2)

with rasterio.open(naip_tif) as src:
    win = rasterio.windows.from_bounds(*neigh_utm, transform=src.transform)
    win = win.round_offsets().round_lengths()
    naip_rgb = src.read([1, 2, 3], window=win)
    naip_rgb = np.clip(naip_rgb, 0, 255).astype(np.uint8)
    naip_rgb = np.transpose(naip_rgb, (1, 2, 0))
    naip_extent = (neigh_utm[0], neigh_utm[2], neigh_utm[1], neigh_utm[3])
    naip_crs = src.crs

with rasterio.open(s2_tif) as src:
    # S2 scene is in its own UTM; reproject neighbourhood bbox first.
    from rasterio.warp import transform_bounds
    s2_neigh = transform_bounds(str(naip_crs), str(src.crs), *neigh_utm)
    win = rasterio.windows.from_bounds(*s2_neigh, transform=src.transform)
    win = win.round_offsets().round_lengths()
    s2_arr = src.read([1, 2, 3], window=win)
    # S2 L2A reflectance scaled by 10000; rescale for visual display
    s2_arr = np.nan_to_num(s2_arr, nan=0.0)
    s2_arr = np.clip(s2_arr / 3000.0, 0, 1)
    s2_arr = np.transpose(s2_arr, (1, 2, 0))   # (H, W, 3)
    s2_extent = (s2_neigh[0], s2_neigh[2], s2_neigh[1], s2_neigh[3])

# Pick a single building polygon near the centre of the neighbourhood
fps = city_footprints[city]
near = fps[fps.geometry.distance(ctr_utm) < NEIGH_M / 2]
if len(near) > 0:
    # Choose the polygon whose centroid is closest to the centre point
    target = near.iloc[near.geometry.centroid.distance(ctr_utm).argmin()]
    t_bb = target.geometry.bounds
else:
    target = None

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(naip_rgb, extent=naip_extent, origin="upper")
axes[0].set_title(f"NAIP, 1 m GSD  ({naip_rgb.shape[1]} x {naip_rgb.shape[0]} px)")
axes[1].imshow(s2_arr, extent=s2_extent, origin="upper")
axes[1].set_title(f"Sentinel-2 L2A, 10 m GSD  ({s2_arr.shape[1]} x {s2_arr.shape[0]} px)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
    if target is not None:
        ax.add_patch(Rectangle((t_bb[0], t_bb[1]),
                               t_bb[2]-t_bb[0], t_bb[3]-t_bb[1],
                               fill=False, edgecolor="red", linewidth=2.0))

# Annotate the size of the marked building in pixels on each panel
if target is not None:
    bw_m = t_bb[2] - t_bb[0]
    bh_m = t_bb[3] - t_bb[1]
    axes[0].set_xlabel(f"red box: {bw_m:.1f} m x {bh_m:.1f} m  -> "
                       f"~{bw_m/1.0:.0f} x {bh_m/1.0:.0f} px at 1 m")
    axes[1].set_xlabel(f"red box: {bw_m:.1f} m x {bh_m:.1f} m  -> "
                       f"~{bw_m/10.0:.1f} x {bh_m/10.0:.1f} px at 10 m")
plt.tight_layout()
plt.show()

print(f"\nA {bw_m:.0f} x {bh_m:.0f} m residential building covers "
      f"{int(round(bw_m))} x {int(round(bh_m))} px on NAIP and "
      f"{bw_m/10.0:.1f} x {bh_m/10.0:.1f} px on Sentinel-2.")
print("YOLO's small-object detection floor is ~16 x 16 px (hard) / ~32 x 32 px "
      "(reliable).")
print("Conclusion: NAIP comfortably clears the floor for residential buildings; "
      "Sentinel-2 falls under it by an order of magnitude.")

## 9. DL: Train a tiny YOLO model on NAIP

We use **YOLOv8n** from Ultralytics -- the "nano" variant, ~3.2 M
parameters, one of the smallest production-grade object detectors that
still trains end-to-end on a CPU in a reasonable time. The architecture
is a single-stage detector: one forward pass through a small CSPDarknet
backbone + a PANet neck + an anchor-free detection head produces all
boxes for the image. Compared with notebook 01's U-Net (which assigns a
class label to every pixel), YOLO's head predicts a sparse list of boxes
plus an objectness score per anchor location.

Training is configured to be CPU-friendly:

| knob | value | reason |
|---|---|---|
| `imgsz` | 512 | matches the tile size we wrote; one resize-free hop |
| `batch` | 4 | low memory pressure on Colab CPUs |
| `epochs` | 80 | needed to get past the 1-class detector's initial plateau |
| `workers` | 2 | matches the OMP threads above |
| `mosaic` | default on | YOLO's standard augmentation; helps with the small dataset |

A `data.yaml` file tells Ultralytics where the images / labels live and
what the class names are.

**On the Ultralytics licence.** Ultralytics is distributed under
AGPL-3.0. Using it as a teaching demo here is fine, but if you embed
YOLO predictions inside a product whose source code you don't intend
to release, you need a commercial licence from Ultralytics or a
different detector altogether (e.g. MMDetection, Detectron2). The
licensing trade-off is the price of how easy this notebook is.

In [ ]:
# Write data.yaml -- absolute paths so we never trip over Ultralytics' cwd.
data_yaml = YOLO_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {YOLO_DIR.resolve()}\n"
    "train: images/train\n"
    "val:   images/val\n"
    "test:  images/test\n"
    "names:\n  0: building\n"
)
print(data_yaml.read_text())

In [ ]:
# Train. Ultralytics writes everything under `project/name/`; we point
# `project=RUNS_DIR` so nothing leaks outside the gitignored scratch folder.
import logging
logging.getLogger("ultralytics").setLevel(logging.WARNING)

t0 = time.time()
model = YOLO("yolov8n.pt")
results = model.train(
    data=str(data_yaml),
    epochs=80,
    imgsz=512,
    batch=4,
    workers=2,
    device="cpu",
    project=str(RUNS_DIR),
    name="building_det",
    exist_ok=True,
    seed=SEED,
    deterministic=True,
    verbose=False,
    pretrained=True,
    save=True,
    save_period=-1,        # only the last + best checkpoint
    plots=True,            # writes png curves we will reuse below
)
TRAIN_SEC = time.time() - t0
print(f"\nTraining wall time: {TRAIN_SEC/60:.1f} min")

Run the standard validation pass at the end and pull out the four
headline metrics. `mAP@0.5` is the average-precision averaged over
classes (we only have one) at a fixed 0.5 IoU threshold; the COCO-style
`mAP@0.5:0.95` averages over IoU thresholds 0.5, 0.55, ... 0.95 and is a
harder, more localisation-sensitive number. Precision and recall are
both reported on the validation set.

In [ ]:
# Run validation on val and test splits
val_metrics  = model.val(data=str(data_yaml), split="val",  imgsz=512,
                         batch=4, device="cpu", verbose=False,
                         project=str(RUNS_DIR), name="val_metrics", exist_ok=True)
test_metrics = model.val(data=str(data_yaml), split="test", imgsz=512,
                         batch=4, device="cpu", verbose=False,
                         project=str(RUNS_DIR), name="test_metrics", exist_ok=True)

def _box_stats(m):
    return {
        "mAP50":     float(m.box.map50),
        "mAP50-95":  float(m.box.map),
        "precision": float(m.box.mp),
        "recall":    float(m.box.mr),
    }

vstats = _box_stats(val_metrics)
tstats = _box_stats(test_metrics)
print(f"{'split':6s} {'mAP@0.5':>8s} {'mAP@0.5:0.95':>14s} {'precision':>10s} {'recall':>8s}")
for split, s in [("val", vstats), ("test", tstats)]:
    print(f"{split:6s} {s['mAP50']:8.3f} {s['mAP50-95']:14.3f} "
          f"{s['precision']:10.3f} {s['recall']:8.3f}")

Plot the training-loss + val-mAP curves directly from the CSV
Ultralytics writes during training (avoids re-running val). This makes
it obvious whether the training is converging or just bouncing around.

In [ ]:
# Ultralytics drops a CSV with one row per epoch under results.csv.
csv_path = RUNS_DIR / "building_det" / "results.csv"
df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

# Common column names: train/box_loss train/cls_loss train/dfl_loss
# metrics/precision(B) metrics/recall(B) metrics/mAP50(B) metrics/mAP50-95(B)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: training losses
loss_cols = [c for c in df.columns if c.startswith("train/") and c.endswith("loss")]
for col in loss_cols:
    axes[0].plot(df["epoch"], df[col], label=col.replace("train/", ""))
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Training losses")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Right: val mAP
map_cols = [c for c in df.columns if c.startswith("metrics/") and "mAP" in c]
for col in map_cols:
    axes[1].plot(df["epoch"], df[col], label=col.replace("metrics/", "").replace("(B)", ""))
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mAP")
axes[1].set_title("Val mAP")
axes[1].set_ylim(0, max(0.05, df[map_cols].max().max() * 1.1))
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Visualise predictions on test tiles

A grid of test tiles with three things drawn:

- the input image,
- **green** ground-truth boxes (from the YOLO label `.txt` files we
  wrote in section 6),
- **magenta** model predictions, each annotated with its objectness
  score.

A YOLO box and a ground-truth box are considered the "same building" if
their **intersection-over-union (IoU)** exceeds 0.5. For each prediction
we annotate the best-IoU match against any ground-truth box so the
reader can eyeball how well-localised each detection is.

In [ ]:
def _iou(a, b):
    """IoU between two (xmin, ymin, xmax, ymax) boxes in pixel coords."""
    ix0 = max(a[0], b[0]); iy0 = max(a[1], b[1])
    ix1 = min(a[2], b[2]); iy1 = min(a[3], b[3])
    iw = max(0.0, ix1 - ix0); ih = max(0.0, iy1 - iy0)
    inter = iw * ih
    aA = (a[2]-a[0]) * (a[3]-a[1])
    aB = (b[2]-b[0]) * (b[3]-b[1])
    union = aA + aB - inter
    return inter / union if union > 0 else 0.0


def _yolo_lines_to_pix(lines, H, W):
    out = []
    for ln in lines:
        parts = ln.split()
        if len(parts) < 5:
            continue
        _cls, cx, cy, w, h = parts[:5]
        cx = float(cx) * W; cy = float(cy) * H
        w  = float(w)  * W; h  = float(h)  * H
        out.append((cx - w/2, cy - h/2, cx + w/2, cy + h/2))
    return out


# Pick six test tiles, biasing toward tiles that have several ground-truth
# buildings so the panel is visually meaningful.
test_img_dir = YOLO_DIR / "images" / "test"
test_lbl_dir = YOLO_DIR / "labels" / "test"
test_imgs = sorted(test_img_dir.glob("*.png"))
test_imgs_with_count = []
for p in test_imgs:
    lp = test_lbl_dir / (p.stem + ".txt")
    n = 0
    if lp.exists():
        n = len([ln for ln in lp.read_text().splitlines() if ln.strip()])
    test_imgs_with_count.append((n, p))
# Sort descending by GT count, take top 12, sample 6
test_imgs_with_count.sort(reverse=True)
candidates = [p for n, p in test_imgs_with_count[:12]]
rng2 = random.Random(SEED + 1)
rng2.shuffle(candidates)
chosen = candidates[:6]

# Run inference once on the chosen tiles
preds = model.predict([str(p) for p in chosen], imgsz=512, device="cpu",
                      conf=0.20, verbose=False, save=False)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, p, pred in zip(axes.flat, chosen, preds):
    img = np.asarray(Image.open(p))
    H, W = img.shape[:2]
    ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])

    # Ground truth in green
    lp = test_lbl_dir / (p.stem + ".txt")
    gt_boxes = _yolo_lines_to_pix(lp.read_text().splitlines() if lp.exists() else [], H, W)
    for (x0, y0, x1, y1) in gt_boxes:
        ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                               edgecolor="lime", linewidth=1.4))
    # Predictions in magenta, with the best-IoU annotation
    p_boxes = []
    p_confs = []
    if pred.boxes is not None and len(pred.boxes) > 0:
        for b in pred.boxes:
            x0, y0, x1, y1 = b.xyxy[0].cpu().numpy().tolist()
            conf = float(b.conf.cpu().numpy())
            p_boxes.append((x0, y0, x1, y1)); p_confs.append(conf)
    for (x0, y0, x1, y1), conf in zip(p_boxes, p_confs):
        best_iou = max([_iou((x0, y0, x1, y1), g) for g in gt_boxes], default=0.0)
        ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                               edgecolor="magenta", linewidth=1.4))
        ax.text(x0, max(0, y0 - 2), f"{conf:.2f} | IoU {best_iou:.2f}",
                fontsize=7, color="magenta",
                bbox=dict(facecolor="white", alpha=0.6, pad=0.5, edgecolor="none"))
    ax.set_title(f"{p.stem}   GT={len(gt_boxes)}, pred={len(p_boxes)}", fontsize=9)

# Legend
green = mpatches.Patch(edgecolor="lime",    facecolor="none", linewidth=1.5, label="ground truth")
mag   = mpatches.Patch(edgecolor="magenta", facecolor="none", linewidth=1.5, label="prediction")
fig.legend(handles=[green, mag], loc="lower center", ncol=2,
           bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.show()

## 11. Sidebar: PlanetScope at 3 m

A natural follow-up is *"what about PlanetScope's 3 m imagery?"*. Two
things make it interesting and one thing makes it awkward.

*Why it's interesting.* PlanetScope's daily-revisit constellation puts
~3 m optical imagery over essentially everywhere on Earth. At 3 m a
10 m residential building occupies ~3 x 3 pixels -- below YOLO's
practical small-object floor for individual houses but comfortably
above it for commercial / industrial / institutional buildings
(typically 30+ m on a side -> ~10 x 10 pixels). The PlanetScope-8b
SuperDove product additionally adds five spectral bands beyond R/G/B
(Coastal Blue, Green I, Yellow, Red Edge, NIR) that can help the
detector tell roofs apart from similarly-coloured paved surfaces.

*Why it's awkward.* Planet imagery **cannot be redistributed publicly**
under the standard research / education licence. That means we can show
the *recipe* in a notebook but cannot embed any pixels in the rendered
output. The pipeline already supports the fetch:

```python
data, bands = fetch_sentinel_data(
    mission="PlanetScope-8b",
    bands=["R", "G", "B", "NIR"],
    time_range=("2024-06-01", "2024-08-31"),
    roi=bbox,
    resolution=3.0,
    save_folder=str(NAIP_DIR.parent / "ps8b"),
)
```

but only runs when `PL_API_KEY` is set in the environment. See the tour
notebook (`00_geoai_datacubes_tour.ipynb`) for how the credentials
guard is wired -- the same pattern works for this notebook: gate the
PlanetScope cell behind `os.getenv("PL_API_KEY")` and add a markdown
caveat that any saved figures should be regenerated locally rather
than committed to the repository.

The natural extension of the workflow above is therefore: run the same
label-conversion + YOLO training, but with a `MIN_BUILDING_M` filter
on the polygon set so the detector trains only on buildings whose
footprints are large enough to span 16+ PlanetScope pixels (~50 m on a
side). The pedagogical point -- *match the imagery resolution to the
object scale* -- is the same one the resolution sidebar in section 8
made.

## 12. Where to go next

You have now seen all three teaching notebooks in the `geoai-datacubes`
series end-to-end:

- **`00_geoai_datacubes_tour.ipynb`** -- the data side. AOI formats,
  per-mission fetches, fusion, tiling, splits, augmentation, on-disk
  export formats.
- **`01_water_classification.ipynb`** -- semantic segmentation. Four
  classifiers (LR / RF / XGB / U-Net) on a binary water-vs-rest target
  derived from ESA WorldCover. Walks through multi-modal fusion
  (S2 -> S2+S1 -> S2+S1+DEM) and cross-city generalisation.
- **`03_building_detection.ipynb`** (this one) -- object detection.
  YOLOv8n on NAIP, with a resolution-comparison sidebar making the case
  that imagery resolution must match the object scale.

Useful next stops:

- **`docs/data_layers.md`** -- the canonical reference for every band
  this pipeline can fetch. The NAIP section in particular explains
  the value range, the multi-band-COG asset layout, and the typical
  normalisation recipe (just divide by 255).
- **GitHub Issue #6** in the repo -- the design discussion behind this
  notebook, including the licensing / variant / class-hierarchy choices
  we made and the ones we deferred.
- **Try a different city.** Swap one of the AOIs in section 2 for a
  city outside Ohio (you will need to download that state's footprints
  file from
  https://github.com/microsoft/USBuildingFootprints#will-there-be-more-data-coming
  ); everything else stays the same.
- **Try a different object class.** Cars, swimming pools, solar panels,
  and centre-pivot irrigation rigs are all natural targets at NAIP
  resolution. You will need a different ground-truth source
  (OpenStreetMap is the common starting point); the rest of the
  workflow -- tile, polygon-to-YOLO, train, evaluate -- is unchanged.
- **Try harder splits.** Right now we train on Columbus, val on
  Cincinnati, test on Cleveland (`regions`-style split). Notebook 00
  shows three other split strategies you could plug in instead.

That's it -- happy detecting.